In [ ]:
!pip install torch torchvision wandb --quiet

In [ ]:
!git clone https://github.com/bremsstrahlung-57/practicum-project /kaggle/working/practicum-project

In [ ]:
%cd /kaggle/working/practicum-project

In [ ]:
from kaggle_secrets import UserSecretsClient
import wandb

secrets = UserSecretsClient()
wandb.login(key=secrets.get_secret("WANDB_API_KEY"))

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet50, ResNet50_Weights
from torch.utils.data import DataLoader

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True,  download=True, transform=train_transform)
test_dataset  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True,  download=True, transform=train_transform)
test_dataset  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')

In [ ]:
def get_cifar_resnet50():
    """
    ResNet-50 adapted for CIFAR-10 (32x32 input).
    Standard ResNet-50 expects 224x224 — two changes fix this:
      1. Replace 7x7 stride-2 conv with 3x3 stride-1 (preserves spatial resolution)
      2. Remove maxpool (would halve 32x32 to 16x16 too aggressively)
    Same adaptation used for ResNet-18 in your other notebooks.
    """
    model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(2048, 10)
    return model.to(DEVICE)

teacher = get_cifar_resnet50()

total_params = sum(p.numel() for p in teacher.parameters()) / 1e6
print(f'ResNet-50 parameters: {total_params:.2f}M')

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += labels.size(0)
    return total_loss / len(loader), 100.0 * correct / total


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        correct += (model(images).argmax(1) == labels).sum().item()
        total   += labels.size(0)
    return 100.0 * correct / total

In [ ]:
EPOCHS = 30
SAVE_PATH = '/kaggle/working/resnet50_cifar10_teacher.pth'

criterion = nn.CrossEntropyLoss()

# Lower LR for pretrained backbone to avoid destroying ImageNet features
# Higher LR for the newly replaced layers (conv1, fc)
new_layers = [teacher.conv1, teacher.fc]
new_layer_ids = {id(p) for layer in new_layers for p in layer.parameters()}

optimizer = torch.optim.SGD([
    {'params': [p for p in teacher.parameters() if id(p) not in new_layer_ids], 'lr': 0.01},
    {'params': [p for p in teacher.parameters() if id(p)     in new_layer_ids], 'lr': 0.1},
], momentum=0.9, weight_decay=5e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

wandb.init(
    project='cnn-compression',
    name='resnet50-teacher-finetune',
    group='distillation',
    config={'epochs': EPOCHS, 'model': 'resnet50', 'task': 'teacher_finetune'}
)

best_acc = 0.0

print(f'Fine-tuning ResNet-50 teacher for {EPOCHS} epochs...')
print(f'{"Epoch":>6} | {"Train Loss":>10} | {"Train Acc":>9} | {"Test Acc":>8}')
print('-' * 45)

for epoch in range(1, EPOCHS + 1):
    loss, train_acc = train_epoch(teacher, train_loader, optimizer, criterion)
    test_acc        = evaluate(teacher, test_loader)
    scheduler.step()

    wandb.log({'epoch': epoch, 'train_loss': loss, 'train_acc': train_acc, 'test_acc': test_acc})

    if epoch % 1 == 0:
        print(f'{epoch:>6} | {loss:>10.4f} | {train_acc:>8.2f}% | {test_acc:>7.2f}%')

    # Save best checkpoint
    if test_acc > best_acc:
        best_acc = test_acc
        torch.save({
            'model_state_dict': teacher.state_dict(),
            'epoch': epoch,
            'accuracy': best_acc
        }, SAVE_PATH)

print(f'\nBest test accuracy: {best_acc:.2f}%')
print(f'Teacher saved to: {SAVE_PATH}')
wandb.finish()

In [ ]:
# Sanity check — reload and re-evaluate
verify_model = get_cifar_resnet50()
ckpt = torch.load(SAVE_PATH, map_location=DEVICE)
verify_model.load_state_dict(ckpt['model_state_dict'])

acc = evaluate(verify_model, test_loader)
print(f'Reloaded teacher accuracy: {acc:.2f}%  (saved at epoch {ckpt["epoch"]})')

In [ ]:
def get_cifar_resnet50():
    model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(2048, 10)
    return model.to(DEVICE)

teacher = get_cifar_resnet50()
ckpt = torch.load('/kaggle/working/resnet50_cifar10_teacher.pth', map_location=DEVICE)
teacher.load_state_dict(ckpt['model_state_dict'])

teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

print(f'Teacher loaded | Epoch {ckpt["epoch"]} | Accuracy: {ckpt["accuracy"]:.2f}%')

In [ ]:
!pip install torch-pruning
from torchvision.models import resnet18
import torch_pruning as tp
def get_cifar_resnet18():
    model = resnet18(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(512, 10)
    return model

def load_structured_pruned_model(ckpt_path, pruning_ratio=0.7):
    model = get_cifar_resnet18().cpu()
    example_input = torch.randn(1, 3, 32, 32)

    importance = tp.importance.MagnitudeImportance(p=1)
    pruner = tp.pruner.MagnitudePruner(
        model          = model,
        example_inputs = example_input,
        importance     = importance,
        pruning_ratio  = pruning_ratio,
        ignored_layers = [model.fc],
    )
    pruner.step()

    ckpt = torch.load(ckpt_path, map_location='cpu')
    model.load_state_dict(ckpt['model_state_dict'])
    return model.to(DEVICE)

student = load_structured_pruned_model(
    '/kaggle/working/practicum-project/models/structured_pruning/pruned/structured_pruned_70pct_fp32.pth',
    pruning_ratio=0.7
)
print('Student loaded')

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        correct += (model(images).argmax(1) == labels).sum().item()
        total   += labels.size(0)
    return 100.0 * correct / total

baseline_acc = evaluate(student, test_loader)
print(f'Pruned student accuracy before distillation: {baseline_acc:.2f}%')

In [ ]:
def distillation_loss(student_logits, teacher_logits, labels, temperature=4.0, alpha=0.7):
    soft_student = torch.log_softmax(student_logits / temperature, dim=1)
    soft_teacher = torch.softmax(teacher_logits  / temperature, dim=1)

    kl_loss = nn.KLDivLoss(reduction='batchmean')(soft_student, soft_teacher)
    kl_loss = kl_loss * (temperature ** 2)  # T^2 rescaling from Hinton et al.

    ce_loss = nn.CrossEntropyLoss()(student_logits, labels)

    return alpha * kl_loss + (1 - alpha) * ce_loss

In [ ]:
def distillation_loss(student_logits, teacher_logits, labels, temperature=4.0, alpha=0.7):
    soft_student = torch.log_softmax(student_logits / temperature, dim=1)
    soft_teacher = torch.softmax(teacher_logits  / temperature, dim=1)

    kl_loss = nn.KLDivLoss(reduction='batchmean')(soft_student, soft_teacher)
    kl_loss = kl_loss * (temperature ** 2)  # T^2 rescaling from Hinton et al.

    ce_loss = nn.CrossEntropyLoss()(student_logits, labels)

    return alpha * kl_loss + (1 - alpha) * ce_loss

In [ ]:
EPOCHS      = 40
TEMPERATURE = 4.0
ALPHA       = 0.7
SAVE_PATH   = '/kaggle/working/resnet18_pruned70_distilled.pth'

optimizer = torch.optim.SGD(student.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_acc = 0.0

print(f'Distillation | T={TEMPERATURE} | alpha={ALPHA} | epochs={EPOCHS}')
print(f'{"Epoch":>6} | {"Loss":>8} | {"Train Acc":>9} | {"Test Acc":>8}')
print('-' * 42)

for epoch in range(1, EPOCHS + 1):
    student.train()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        with torch.no_grad():
            teacher_logits = teacher(images)

        student_logits = student(images)
        loss = distillation_loss(student_logits, teacher_logits, labels, TEMPERATURE, ALPHA)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct    += (student_logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)

    scheduler.step()
    test_acc  = evaluate(student, test_loader)
    train_acc = 100.0 * correct / total

    print(f'{epoch:>6} | {total_loss/len(train_loader):>8.4f} | {train_acc:>8.2f}% | {test_acc:>7.2f}%')

    if test_acc > best_acc:
        best_acc = test_acc
        torch.save({
            'model_state_dict': student.state_dict(),
            'epoch': epoch,
            'accuracy': best_acc,
            'temperature': TEMPERATURE,
            'alpha': ALPHA,
        }, SAVE_PATH)

print(f'\nBefore distillation: {baseline_acc:.2f}%')
print(f'After distillation:  {best_acc:.2f}%')
print(f'Delta:               {best_acc - baseline_acc:+.2f}%')